# Module 3: Python for Data Engineer

**EJ** · Stage 3 · Nov 19–23, 2026 · image: python-training:3.0

## Section 3.1: Datetime

### B-3.1.1

Create 30 timestamp strings in three formats, parse each group with an explicit format, and count the NaTs produced by a deliberately bad value (Lab 3.1 step 1).

In [2]:
# Will use the existing timestamps.csv file

import pandas as pd

df = pd.read_csv("../data/module3/timestamps.csv")

print(df.head())
print("Total rows:", len(df))

   event_id source_system               ts_raw
0         1           crm     2026-03-01 08:00
1         2           erp           02/03/2026
2         3           web  Mar 3, 2026 8:00 AM
3         4           crm     2026-03-03 11:00
4         5           erp           04/03/2026
Total rows: 30


The file has three columns: **event_id, source_system, and ts_raw** and 30 rows.

There are three source system groups: **crm, erp, and web**

In [3]:
# The explicit formats for each group

formats = {
    "crm": "%Y-%m-%d %H:%M",
    "erp": "%d/%m/%Y",
    "web": "%b %d, %Y %I:%M %p",
}

In [4]:
df["ts_parsed"] = pd.NaT # a new column called ts_parsed initially filled with NaT

# Stop if a source_system has no format, so its rows aren't miscounted as bad
assert df["source_system"].isin(formats).all(), "Unknown source_system found"

# Parsing each group with the format
for source_system, ts_format in formats.items():
    rows = df["source_system"] == source_system # Returns a True or False if it's a match

    df.loc[rows, "ts_parsed"] = pd.to_datetime(df.loc[rows, "ts_raw"], format=ts_format, errors="coerce")

In [12]:
df

,event_id,source_system,ts_raw,ts_parsed,ts_local,ts_utc
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00
5,6,web,"Mar 5, 2026 11:00 AM",2026-03-05 11:00:00,2026-03-05 11:00:00+08:00,2026-03-05 03:00:00+00:00
6,7,crm,2026-03-05 14:00,2026-03-05 14:00:00,2026-03-05 14:00:00+08:00,2026-03-05 06:00:00+00:00
7,8,erp,31/02/2026,NaT,NaT,NaT
8,9,web,"Mar 7, 2026 2:00 PM",2026-03-07 14:00:00,2026-03-07 14:00:00+08:00,2026-03-07 06:00:00+00:00
9,10,crm,2026-03-07 17:00,2026-03-07 17:00:00,2026-03-07 17:00:00+08:00,2026-03-07 09:00:00+00:00


In [11]:
bad = df[df["ts_parsed"].isna()] # returns only the rows where ts_parsed is NaT
bad

,event_id,source_system,ts_raw,ts_parsed,ts_local,ts_utc
7,8,erp,31/02/2026,NaT,NaT,NaT
20,21,web,"Marz 9, 2026 3:00 PM",NaT,NaT,NaT


Among the 30 rows, only two are not valid timestamps:

- event_id 8 - **31/02/2026**: there is no february 31
- event_id 21 - **Marz 9, 2026** 3:00 PM: Marz is not a valid month name

### B-3.1.2

Make the datetimes timezone-aware, convert to UTC, and show both side by side (Lab 3.1 step 2).

In [7]:
# Assigning a local timezone
df["ts_local"] = df["ts_parsed"].dt.tz_localize("Asia/Manila")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00


In [8]:
# Converting the local UTC+8 to UTC+0
df["ts_utc"] = df["ts_local"].dt.tz_convert("UTC")
df.head()

,event_id,source_system,ts_raw,ts_parsed,ts_local,ts_utc
0,1,crm,2026-03-01 08:00,2026-03-01 08:00:00,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,erp,02/03/2026,2026-03-02 00:00:00,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,web,"Mar 3, 2026 8:00 AM",2026-03-03 08:00:00,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,crm,2026-03-03 11:00,2026-03-03 11:00:00,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,erp,04/03/2026,2026-03-04 00:00:00,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


In [10]:
# The two timezone's side-by-side

df[["event_id", "ts_local", "ts_utc"]].head()

,event_id,ts_local,ts_utc
0,1,2026-03-01 08:00:00+08:00,2026-03-01 00:00:00+00:00
1,2,2026-03-02 00:00:00+08:00,2026-03-01 16:00:00+00:00
2,3,2026-03-03 08:00:00+08:00,2026-03-03 00:00:00+00:00
3,4,2026-03-03 11:00:00+08:00,2026-03-03 03:00:00+00:00
4,5,2026-03-04 00:00:00+08:00,2026-03-03 16:00:00+00:00


For event_id 2, **March 2 at 00:00** in Manila or 12 AM becomes **March 1 at 16:00** or 4PM in UTC.

Because Manila is **UTC+8**, any local time before 8:00 AM falls on the previous day in UTC

### B-3.1.3

Derive year, month, weekday, and hour columns, and resample to daily counts (Lab 3.1 step 3).

In [15]:
# Using the same ts_local so it follows local time

df["year"] = df["ts_local"].dt.year
df["month"] = df["ts_local"].dt.month
df["weekday"] = df["ts_local"].dt.day_name()
df["hour"] = df["ts_local"].dt.hour

df[["event_id", "ts_local", "year", "month", "weekday", "hour"]].head() 

,event_id,ts_local,year,month,weekday,hour
0,1,2026-03-01 08:00:00+08:00,2026.0,3.0,Sunday,8.0
1,2,2026-03-02 00:00:00+08:00,2026.0,3.0,Monday,0.0
2,3,2026-03-03 08:00:00+08:00,2026.0,3.0,Tuesday,8.0
3,4,2026-03-03 11:00:00+08:00,2026.0,3.0,Tuesday,11.0
4,5,2026-03-04 00:00:00+08:00,2026.0,3.0,Wednesday,0.0


Since there are two NaT values, the output shows float versions for the year, month, and hour. Missing values are stored as NaN, which is a float, so pandas turns the whole column into float to fit them.

In [22]:
# Resampling to daily counts

daily_counts = (df.resample("D", on="ts_local").size().rename("event_count"))
# D groups events into calendar days using ts_local

print("Days:", len(daily_counts))
daily_counts

Days: 22


ts_local
2026-03-01 00:00:00+08:00    1
2026-03-02 00:00:00+08:00    1
2026-03-03 00:00:00+08:00    2
2026-03-04 00:00:00+08:00    1
2026-03-05 00:00:00+08:00    2
2026-03-06 00:00:00+08:00    0
2026-03-07 00:00:00+08:00    2
2026-03-08 00:00:00+08:00    1
2026-03-09 00:00:00+08:00    2
2026-03-10 00:00:00+08:00    1
2026-03-11 00:00:00+08:00    2
2026-03-12 00:00:00+08:00    1
2026-03-13 00:00:00+08:00    1
2026-03-14 00:00:00+08:00    1
2026-03-15 00:00:00+08:00    1
2026-03-16 00:00:00+08:00    1
2026-03-17 00:00:00+08:00    1
2026-03-18 00:00:00+08:00    2
2026-03-19 00:00:00+08:00    1
2026-03-20 00:00:00+08:00    2
2026-03-21 00:00:00+08:00    1
2026-03-22 00:00:00+08:00    1
Name: event_count, dtype: int64

There are **28 valid events over 22 days** (the 2 invalid timestamps are excluded). Mar 6 shows 0 because resample lists every calendar day in the range, even days with no events.

### B-3.1.4

Compute the time between two events with `timedelta` (Lab 3.1 step 4).

In [30]:
# Will use the evets.csv file for this, specifically the ordered_at and shipped_at columns

events = pd.read_csv("../data/module3/events.csv")
iso_format = "%Y-%m-%dT%H:%M:%S" # original format of the timestamp

# Parse the format into the ordered_at and shipped_at events
for column in ["ordered_at", "shipped_at"]:
    events[column] = pd.to_datetime(events[column], format=iso_format, errors="coerce")

print(events.shape)
events.head()

(10, 3)


,event_id,ordered_at,shipped_at
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00
4,E005,2026-03-06 15:45:00,NaT


The file has **three columns and 10 rows**. One row has a **missing** shipped_at value.

In [31]:
# timedelta handles a missing value (event 5)
events["order_to_ship_duration"] = events["shipped_at"] - events["ordered_at"]

events

,event_id,ordered_at,shipped_at,order_to_ship_duration
0,E001,2026-03-02 15:45:00,2026-03-03 18:45:00,1 days 03:00:00
1,E002,2026-03-03 10:45:00,2026-03-06 09:45:00,2 days 23:00:00
2,E003,2026-03-04 16:15:00,2026-03-06 09:15:00,1 days 17:00:00
3,E004,2026-03-05 13:45:00,2026-03-10 01:45:00,4 days 12:00:00
4,E005,2026-03-06 15:45:00,NaT,NaT
5,E006,2026-03-07 15:15:00,2026-03-09 04:15:00,1 days 13:00:00
6,E007,2026-03-08 08:00:00,2026-03-11 04:00:00,2 days 20:00:00
7,E008,2026-03-09 10:15:00,2026-03-09 19:15:00,0 days 09:00:00
8,E009,2026-03-10 17:15:00,2026-03-11 00:15:00,0 days 07:00:00
9,E010,2026-03-11 17:00:00,2026-03-15 14:00:00,3 days 21:00:00


In [33]:
print("Average:", events["order_to_ship_duration"].mean())
print("Longest:", events["order_to_ship_duration"].max())

Average: 2 days 03:13:20
Longest: 4 days 12:00:00


E005 has no **shipped_at**, so it parses to NaT. Any subtraction with NaT gives NaT, so its order_to_ship_duration is NaT too. 

**.mean() and .max() skip NaT**, which means the average covers 9 orders, not 10.

## Section 3.2: JSON, CSV, and Excel

### B-3.2.1

Create a small nested JSON file, flatten it to one row per item, and save it to CSV and Excel (Lab 3.2 step 1).

### B-3.2.2

Read the CSV back with explicit `dtype` and `usecols`, read the Excel file back, and compare the three versions (Lab 3.2 step 2).

### B-3.2.3

Read a larger CSV in chunks and compute a running total (Lab 3.2 step 3).

## Section 3.3: Avro

### B-3.3.1

Define an Avro schema for the flattened order-item table with one optional field (Lab 3.3 step 1).

### B-3.3.2

Write `orders.avro`, read it back, and load it into a DataFrame (Lab 3.3 step 2).

### B-3.3.3

Compare file sizes of CSV, JSON, and Avro for the same data in a table (Lab 3.3 step 3).

### B-3.3.4

Add a field to the schema and show reading old data with the new schema (Lab 3.3 step 4).

## Section 3.4: Web scraping (local sandbox)

### B-3.4.1

Add the `sandbox-site` service under the `scrape` profile and start it (Lab 3.4 step 1).

### B-3.4.2

Fetch and parse the site's `robots.txt`, and write down what it allows for the trainee's user agent and the date checked (Lab 3.4 step 2).

### B-3.4.3

Scrape one listing page, extracting at least 4 fields per item (Lab 3.4 step 3).

### B-3.4.4

Follow pagination across all pages with a delay and a descriptive user-agent string (Lab 3.4 step 4).

### B-3.4.5

Add a check that skips disallowed URLs and show it refusing the `/admin/` path (Lab 3.4 step 5).

### B-3.4.6

Clean the results (price to float) and save them to `data/products.csv` (Lab 3.4 step 6).

## Diagrams

## What broke and how I fixed it

## Reflection

A reflection of 3 to 5 sentences